In [14]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/train_spider.json
/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/train_others.json
/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/dev_gold.sql
/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/dev.json
/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/README.txt
/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/train_gold.sql
/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/tables.json
/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/database/solvency_ii/solvency_ii.sqlite
/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/database/solvency_ii/schema.sql
/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset

# Loading dataset

In [15]:
from datasets import load_dataset #loaded into the cpu. # dataset is a library and load_dataset is a function (as it has lowercase with underscore)

dataset = load_dataset("spider")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 7000
    })
    validation: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 1034
    })
})


In [16]:
#loading the schema of the dbs

spider_tables = load_dataset("richardr1126/spider-schema", split="train")
print(spider_tables)

schema_lookup ={}
for row in spider_tables:
  schema_lookup[row['db_id']]= row
schema_lookup['concert_singer']


Dataset({
    features: ['db_id', 'Schema (values (type))', 'Primary Keys', 'Foreign Keys'],
    num_rows: 166
})


{'db_id': 'concert_singer',
 'Schema (values (type))': 'stadium : Stadium_ID (number) , Location (text) , Name (text) , Capacity (number) , Highest (number) , Lowest (number) , Average (number) | singer : Singer_ID (number) , Name (text) , Country (text) , Song_Name (text) , Song_release_year (text) , Age (number) , Is_male (others) | concert : concert_ID (number) , concert_Name (text) , Theme (text) , Stadium_ID (text) , Year (text) | singer_in_concert : concert_ID (number) , Singer_ID (text)',
 'Primary Keys': 'stadium : Stadium_ID | singer : Singer_ID | concert : concert_ID | singer_in_concert : concert_ID',
 'Foreign Keys': 'concert : Stadium_ID equals stadium : Stadium_ID | singer_in_concert : Singer_ID equals singer : Singer_ID | singer_in_concert : concert_ID equals concert : concert_ID'}

In [17]:
schema_lookup['activity_1']

{'db_id': 'activity_1',
 'Schema (values (type))': 'Activity : actid (number) , activity_name (text) | Participates_in : stuid (number) , actid (number) | Faculty_Participates_in : FacID (number) , actid (number) | Student : StuID (number) , LName (text) , Fname (text) , Age (number) , Sex (text) , Major (number) , Advisor (number) , city_code (text) | Faculty : FacID (number) , Lname (text) , Fname (text) , Rank (text) , Sex (text) , Phone (number) , Room (text) , Building (text)',
 'Primary Keys': 'Activity : actid | Student : StuID | Faculty : FacID',
 'Foreign Keys': 'Participates_in : actid equals Activity : actid | Participates_in : stuid equals Student : StuID | Faculty_Participates_in : actid equals Activity : actid | Faculty_Participates_in : FacID equals Faculty : FacID'}

## utils: parsing functions

In [18]:
def parse_primary_keys(pk_string):

  pk_map ={}
  # There are database full of tables but has not primary keys at all.
  if not pk_string:
    return pk_map

  for entry in pk_string.split('|'):
    entry = entry.strip()
    table, cols = entry.split(':')

    pk_map[table.strip()] =cols.strip()
  return pk_map

db = 'concert_singer'
map = parse_primary_keys(pk_string = schema_lookup[db]['Primary Keys'])
print(map)

{'stadium': 'Stadium_ID', 'singer': 'Singer_ID', 'concert': 'concert_ID', 'singer_in_concert': 'concert_ID'}


In [19]:
def parse_foreign_keys(fk_string):
    fk_map = {}
    words = fk_string
    fk_string = words.split()
    for i,word in enumerate(fk_string):
        if word =='equals':
            first_table = fk_string[i-3]
            second_table = fk_string[i+1]

            fk_map[(first_table, fk_string[i-1])] = (second_table, fk_string[i+3])


    return fk_map
map = parse_foreign_keys(fk_string = schema_lookup[db]['Foreign Keys'])
print(map)

{('concert', 'Stadium_ID'): ('stadium', 'Stadium_ID'), ('singer_in_concert', 'Singer_ID'): ('singer', 'Singer_ID'), ('singer_in_concert', 'concert_ID'): ('concert', 'concert_ID')}


# Creating col level metadata for each db

In [20]:
def parse_schema(db_id):
    """
    'col_dicts': [
        {
          'table_name': ...,
          'column_name': ...,
          'column_type': ...,
          'embed_text': 'a column named X in table Y',
          'statistics': ...,
          'is_primary_key': bool,
          'foreign_key': None or {'ref_table': ..., 'ref_column': ...}
        },
        ...
      ]
    """   

    row = schema_lookup[db_id]
    schema = row['Schema (values (type))']
    pk_map = parse_primary_keys(row['Primary Keys'])
    fk_map = parse_foreign_keys(row['Foreign Keys'])

    col_dicts = [] 
    
    for table_info in schema.split('|'):

        table_info = table_info.strip()
        
        table_name, cols_info = table_info.split(':')
        table_name = table_name.strip()
        cols_info = cols_info.strip()

        # a table can have no primary key.
        pk =''
        if table_name in pk_map:
            pk = pk_map[table_name]

        for col_info in cols_info.split(','):

            col_info = col_info.strip()

            col_name = col_info[:col_info.rfind('(')].strip()
            col_type = col_info[col_info.rfind('(')+1 : col_info.rfind(')')].strip()

            # checking if foreign key exist for this column
            fk_found = False
            col_to_find = col_name

            if (table_name, col_to_find) in fk_map:
                mapping = fk_map[(table_name, col_to_find)]
                fk_found = True

            temp_dic ={}
            temp_dic['table_name'] = table_name
            temp_dic['column_name'] = col_name
            temp_dic['column_type'] = col_type
            temp_dic['is_primary_key'] = True if pk and col_name == pk else False
            temp_dic['foreign_key'] = None
            
            if fk_found:

                temp = {}
                temp['ref_table'] = mapping[0]
                temp['ref_column'] = mapping[1]
                temp_dic['foreign_key'] = temp

            col_dicts.append(temp_dic)
    return col_dicts

db_id = db
col_dicts = parse_schema(db_id)
print(col_dicts)


[{'table_name': 'stadium', 'column_name': 'Stadium_ID', 'column_type': 'number', 'is_primary_key': True, 'foreign_key': None}, {'table_name': 'stadium', 'column_name': 'Location', 'column_type': 'text', 'is_primary_key': False, 'foreign_key': None}, {'table_name': 'stadium', 'column_name': 'Name', 'column_type': 'text', 'is_primary_key': False, 'foreign_key': None}, {'table_name': 'stadium', 'column_name': 'Capacity', 'column_type': 'number', 'is_primary_key': False, 'foreign_key': None}, {'table_name': 'stadium', 'column_name': 'Highest', 'column_type': 'number', 'is_primary_key': False, 'foreign_key': None}, {'table_name': 'stadium', 'column_name': 'Lowest', 'column_type': 'number', 'is_primary_key': False, 'foreign_key': None}, {'table_name': 'stadium', 'column_name': 'Average', 'column_type': 'number', 'is_primary_key': False, 'foreign_key': None}, {'table_name': 'singer', 'column_name': 'Singer_ID', 'column_type': 'number', 'is_primary_key': True, 'foreign_key': None}, {'table_nam

# Computing statistics for each column of all tables in db

In [21]:
import sqlite3
import os

def compute_statistics(col_dicts, db_id, db_base_path):
    """
    Takes col_dicts from step 1, connects to the SQLite DB,
    and adds 'statistics' field to each dict in place.
    
    db_base_path: path to the 'database' folder
    e.g: '/kaggle/input/spider/database'
    """
    db_path = os.path.join(db_base_path, db_id, f"{db_id}.sqlite")
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    for col in col_dicts:
        table = col['table_name']
        column = col['column_name']
        
        try:
            cursor.execute(
                f"SELECT DISTINCT `{column}` FROM `{table}` "  #Backtick quoting around column and table names — necessary because some Spider column names are reserved SQL words (Year, Name, Average) that will throw a syntax error without quoting
                f"WHERE `{column}` IS NOT NULL LIMIT 3"
            )
            rows = cursor.fetchall()
            values = [row[0] for row in rows]
            col['statistics'] = f"e.g: {values}"
        
        except Exception as e:
            col['statistics'] = "N/A"
    
    conn.close()
    return col_dicts

db_id = db
col_dicts = compute_statistics(col_dicts, db_id, db_base_path = '/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/database' )
print(col_dicts[10])


{'table_name': 'singer', 'column_name': 'Song_Name', 'column_type': 'text', 'is_primary_key': False, 'foreign_key': None, 'statistics': "e.g: ['You', 'Dangerous', 'Hey Oh']"}


In [22]:
# from sentence_transformers import SentenceTransformer
# # Load once globally, not inside the function
# model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')



# Creating schema (table + fk relationship) for each db (broader topology for model to traverse)

In [23]:
def build_schema(col_dicts):

    schema = ''

    check_table= {}
    table_str = ''
    Fk_str = ''

    for col_meta in col_dicts:

        table_name = col_meta['table_name']
        col_name = col_meta['column_name']

        if table_name not in check_table:

            check_table[table_name] = 1

            if not table_str:
                table_str+= table_name
            else:
                table_str+= f', {table_name}'

        
        if col_meta['foreign_key']:

            ref_table = col_meta['foreign_key']['ref_table']
            ref_column = col_meta['foreign_key']['ref_column']

            text = f'{table_name}.{col_name} = {ref_table}.{ref_column}'

            if not Fk_str:
                Fk_str+= text
            else:
                Fk_str+= f'\n{text}'
            
    schema = f'Tables: {table_str}\nForeignKeys:\n{Fk_str}'
    return schema

In [24]:

schema = build_schema(col_dicts)
print(schema)

Tables: stadium, singer, concert, singer_in_concert
ForeignKeys:
concert.Stadium_ID = stadium.Stadium_ID
singer_in_concert.concert_ID = concert.concert_ID
singer_in_concert.Singer_ID = singer.Singer_ID


In [25]:
import json
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("Openai") 

client = OpenAI(api_key=api_key)

# Computing column embeddings for each db

In [29]:
from sklearn.preprocessing import normalize
def compute_embeddings(col_dicts):
    """
    Takes col_dicts (with statistics already computed).
    Adds 'embed_text' to each dict.
    Returns embedding matrix of shape (n_cols, 384).
    """

    text_to_embed = []
    for col in col_dicts:
        
        col_name = col['column_name'].replace('_', ' ').lower() #Singer_ID as "singer id"
        table_name = col['table_name'].replace('_', ' ').lower() 
    
        embed_text = f"a column named {col_name} in table {table_name}."
        col['embed_text'] = embed_text
        
        text_to_embed.append(embed_text)

    response = client.embeddings.create(
    model="text-embedding-3-small",
    input= text_to_embed
)
    embeddings= np.array([d.embedding for d in response.data])   # same order as input
    embeddings_norm = normalize(embeddings, norm='l2')
    

    # embeddings = model.encode(text_to_embed, normalize_embeddings = True)
    return embeddings_norm

embeddings = compute_embeddings(col_dicts)
embeddings.shape

(21, 1536)

In [30]:
col_dicts[11]

{'table_name': 'singer',
 'column_name': 'Song_release_year',
 'column_type': 'text',
 'is_primary_key': False,
 'foreign_key': None,
 'statistics': "e.g: ['1992', '2008', '2013']",
 'embed_text': 'a column named song release year in table singer.'}

# Building graph for each db

In [31]:
import networkx as nx
from itertools import combinations
from collections import defaultdict

def build_graph(col_dicts):

    G = nx.Graph()

    table_to_column_map = defaultdict(list)
    
    for col_dic in col_dicts:

        node = f"{col_dic['table_name']}.{col_dic['column_name']}" # olumns with no FK and are the only column in their table won't get added via add_edge. For safety added
        G.add_node(node)
        
        table = col_dic['table_name']
        col = col_dic['column_name']
        
        if col_dic['foreign_key']:
            
            ref_table = col_dic['foreign_key']['ref_table']
            ref_column = col_dic['foreign_key']['ref_column']

            start = f'{table}.{col}'
            end = f'{ref_table}.{ref_column}'
            
            G.add_edge(start, end)


        table_to_column_map[table].append(col) 

    
    for table, col_list in table_to_column_map.items():

        pairs = combinations(col_list, 2)

        for pair in pairs:

            start = f'{table}.{pair[0]}'
            end = f'{table}.{pair[1]}'

            G.add_edge(start, end)

    return G
        

# computing for all db

In [32]:
for db_id, values in schema_lookup.items():
    
    col_dicts = parse_schema(db_id)
    col_dicts = compute_statistics(col_dicts, db_id, db_base_path = '/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/database' )
    embeddings = compute_embeddings(col_dicts)
    schema = build_schema(col_dicts)

    schema_lookup[db_id]['embeddings'] = embeddings
    schema_lookup[db_id]['col_dicts'] = col_dicts
    schema_lookup[db_id]['graph'] = build_graph(col_dicts)
    schema_lookup[db_id]['schema'] = schema
    
    

In [33]:
db = schema_lookup['concert_singer']
G = db['graph']

print("Nodes:", G.number_of_nodes())  # should be 21
print("Edges:", G.number_of_edges())

# Check FK edges specifically
print("\nFK edges:")
for u, v in G.edges():
    # FK edges cross tables
    if u.split('.')[0] != v.split('.')[0]:
        print(f"  {u} <-> {v}")

Nodes: 21
Edges: 56

FK edges:
  stadium.Stadium_ID <-> concert.Stadium_ID
  singer.Singer_ID <-> singer_in_concert.Singer_ID
  concert.concert_ID <-> singer_in_concert.concert_ID


# Saving the embeddings, graph, and schema for each db 

In [34]:
# Pickle to disk
import pickle
with open('spider_schema_lookup.pkl', 'wb') as f:
    pickle.dump(schema_lookup, f)

print(f"\nSaved schema_lookup with {len(schema_lookup)} DBs")



Saved schema_lookup with 166 DBs


In [35]:
print(schema_lookup['car_1']['schema'])

Tables: continents, countries, car_makers, model_list, car_names, cars_data
ForeignKeys:
countries.Continent = continents.ContId
car_makers.Country = countries.CountryId
model_list.Maker = car_makers.Id
car_names.Model = model_list.Model
cars_data.Id = car_names.MakeId


In [36]:
print(schema_lookup['academic']['schema'])

Tables: author, conference, domain, domain_author, domain_conference, journal, domain_journal, keyword, domain_keyword, publication, domain_publication, organization, publication_keyword, writes, cite
ForeignKeys:
domain_author.aid = author.aid
domain_author.did = domain.did
domain_conference.cid = conference.cid
domain_conference.did = domain.did
domain_journal.did = domain.did
domain_journal.jid = journal.jid
domain_keyword.did = domain.did
domain_keyword.kid = keyword.kid
publication.cid = conference.cid
publication.jid = journal.jid
domain_publication.did = domain.did
domain_publication.pid = publication.pid
publication_keyword.pid = publication.pid
publication_keyword.kid = keyword.kid
writes.aid = author.aid
writes.pid = publication.pid
cite.cited = publication.pid
cite.citing = publication.pid
